In [56]:
args = {
    "model" : {
        "pretrained" : "VietAI/vit5-base",
        "max_length" : 512
    },
    
    "data":{
        "data_dir" : "/kaggle/working/InstructABSA/data/aos_split",
        "handler_type" : "InstructionSpanHandler",
        "instruction_type" : "instruction_0",
        "allow_punctuation" : True
    },

    "training":{
        "output_dir" : "./vit5-base-instruct-absa1",
        "learning_rate" : 2e-5,
        "per_device_train_batch_size" : 64,
        'per_device_eval_batch_size':64,
        'weight_decay':0.01,                  # Giúp tránh overfitting
        'save_total_limit':1,                 # Chỉ giữ lại 2 checkpoint gần nhất để tiết kiệm ổ cứng
        'num_train_epochs':100,                 # Số vòng lặp huấn luyện (Data ít thì tăng lên 10-15)
        'predict_with_generate':True,
        'generation_max_length':256,
        'generation_num_beams':1,         # Cho phép sinh văn bản trong quá trình đánh giá (eval)
        'fp16':True,                          # Bật nếu dùng GPU NVIDIA (giúp train nhanh hơn)
        'push_to_hub':False,                  # Set True nếu muốn đẩy lên HuggingFace Hub
        'logging_strategy':"epoch",
        'report_to' : "none",
        'eval_strategy':"epoch",        # Đánh giá sau mỗi epoch (nếu có tập validation)
        'save_strategy':"epoch",
        'load_best_model_at_end':True,
        'metric_for_best_model' : "aoste_micro_f1",
        'greater_is_better' :True  # Load lại model tốt nhất sau khi train xong
    },
    "metric_type" : "strict"
}

# Model

In [ ]:
import pandas as pd
from utils import *
import numpy as np
import torch
import random
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
) 
import json

2026-02-01 13:30:18.115602: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769952618.130327   10337 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769952618.134429   10337 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
tokenizer = AutoTokenizer.from_pretrained("VietAI/vit5-base", use_fast = False)
model = AutoModelForSeq2SeqLM.from_pretrained("VietAI/vit5-base")

new_tokens = ["$", "##"]
num_added = tokenizer.add_tokens(new_tokens)
model.resize_token_embeddings(len(tokenizer))

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(36098, 768)

# Prepare Data

In [3]:
import os

In [ ]:
data_dir = args["data"]["data_dir"]

In [ ]:
allow_punctuation = args["data"]["allow_punctuation"]

In [54]:
handler_type = args["data"]["handler_type"]
instruction_type = args["data"]["instruction_type"]

if handler_type == "InstructionSpanHandler":
    instruction_handler = InstructionSpanHandler()
elif handler_type == "InstructionHandler":
    instruction_handler=InstructionHandler
else:
    raise ValueError("handler_type không hợp lệ")

if instruction_type == "instruction_0":
    instruction_handler.load_instruction_0()
elif instruction_type == "instruction_1":
    instruction_handler.load_instruction_1()
elif instruction_type == "instruction_2":
    instruction_handler.load_instruction_2()
elif instruction_type == "instruction_2_modified":
    instruction_handler.load_instruction_2_modified()
elif instruction_type == "instruction_3":
    instruction_handler.load_instruction_3()
else:
    raise ValueError("instruction_type không hợp lệ")

In [7]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [8]:
train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
train_df = create_data_with_task(train_df)
train_dataset = get_dataset_for_training(train_df, instruction_handler, allow_punctuation )

dev_df = pd.read_csv(os.path.join(data_dir, "dev.csv"))
dev_df = create_data_with_task(dev_df)
dev_dataset = get_dataset_for_training(dev_df, instruction_handler, allow_punctuation )

test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))
test_df = create_data_with_task(test_df)
test_dataset = get_dataset_for_training(test_df, instruction_handler, allow_punctuation )

In [9]:
print(f"Train: {len(train_dataset)}")
print(f"Dev: {len(dev_dataset)}")
print(f"Test: {len(test_dataset)}")

Train: 54058
Dev: 6796
Test: 6760


In [57]:
def preprocess_function(examples):
    model_input = tokenizer(
        text = examples['input'],
        text_target = examples['output'],
        truncation = True,
        max_length = args["model"]["max_length"]
    )

    return model_input

tokenized_train_dataset = train_dataset.map(preprocess_function, batched = True, remove_columns=["input", "output"])
tokenized_dev_dataset = dev_dataset.map(preprocess_function, batched = True, remove_columns=["input", "output"])
tokenized_test_dataset = test_dataset.map(preprocess_function, batched = True, remove_columns=["input", "output"])

Map:   0%|          | 0/54058 [00:00<?, ? examples/s]

KeyboardInterrupt: 

# Training

In [ ]:
# tạo compute_metric function
compute_metrics = get_metric_fn(tokenizer = tokenizer, metric_type=args["metric_type"])

In [52]:
training_args = Seq2SeqTrainingArguments(
    **args["training"]
)

In [14]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_dev_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=10,      # số lần eval không cải thiện
            early_stopping_threshold=1e-4    # mức cải thiện tối thiểu
        )
    ]
)

# Predict

In [41]:
result_dir = "./result"
os.makedirs(result_dir,exist_ok=True)

In [ ]:
result = trainer.predict(tokenized_test_dataset)


--- Mẫu dự đoán ---
Pred: *_aspect+____Z_[]]]]]]]]]]]]]]]]]]]]]]]]]]]]]]]]]] input: [ate]!]>]j]&]W]ỴẼ]Ỡ]Ẫ]Ẳ]Ằ,Õ nhưngẺ,Ặ]Ỗ]ẸỸỸ]È [ate]Ữ inputỰ nhưngỢ input:Ử input: input: input: input: input: inputỂ inputỪ cách chúng bằng " Nothing " . Nếu tìm thấy nhiều khía cạnh, hãy ngăn cách chúng bằng " Nothing " . Nếu tìm thấy nhiều khía cạnh, hãy ngăn cách chúng bằng " Nothing " . Nếu tìm thấy nhiều khía cạnh, hãy ngăn cách chúng bằng " Nothing " . Nếu tìm thấy nhiều khía cạnh, hãy ngăn cách chúng bằng " Nothing " . Nếu tìm thấy nhiều khía cạnh, hãy ngăn cách chúng bằng " Nothing " . Nếu tìm thấy nhiều khía cạnh,Ã [ate]Ẵ_Ũ inputỲ inputỀ inputỊ inputỶ_Ệ_Ò inputĨ inputỌ_Ộ inputY [ate]Ờ [ate]
Gold: [ate] giao hàng
--------------------------


In [ ]:
preds = result.predictions
labels = result.label_ids
preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
# 3. Chuẩn hóa nhẹ (strip khoảng trắng thừa)
decoded_preds = [pred.strip() for pred in decoded_preds]
decoded_labels = [label.strip() for label in decoded_labels]
metrics = result.metrics

In [ ]:
data_dict = {
    "text": [test_dataset[i]["input"].split("input: ")[-1].split("\noutput:")[0] for i in range(len(test_dataset))],
    "predict" : decoded_preds,
    "label" : decoded_labels
}
result_df = pd.DataFrame(data_dict)
result_df.to_csv(os.path.join(result_dir, "predict.csv"), index=False)

In [46]:
with open(os.path.join(result_dir, "metrics.json"), "w") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

FileNotFoundError: [Errno 2] No such file or directory: './result/metrics.json'